In [1]:
# Instalación de librerías necesarias
!pip install -q -U langchain-groq langchain-community langgraph ultralytics langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.3/496.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
from google.colab import userdata
from google.colab import files
from IPython.display import Image, display
import getpass

# Configuración de la API Key de Groq
# Intenta obtener la clave de los Secretos de Colab
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("API Key cargada desde los secretos de Colab.")
except Exception as e:
    # Si falla (o no existe el secreto), te la pedirá manualmente
    print("No se encontró el secreto 'GROQ_API_KEY'.")
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Introduce tu Groq API Key manualmente: ")

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent
from ultralytics import YOLO
from collections import Counter
import warnings

API Key cargada desde los secretos de Colab.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Inicializamos el modelo YOLO

yolo_model = YOLO('yolov8n.pt')
print("Modelo YOLO cargado y listo")

Modelo YOLO cargado y listo


In [4]:
@tool
def detectar_objetos(ruta_imagen: str) -> str:
    """
    Utiliza esta herramienta para detectar qué objetos, animales o personas hay en una imagen.
    Entrada: La ruta del archivo de la imagen (ej: 'imagen.jpg').
    Salida: Un resumen de texto con los objetos detectados y su cantidad.
    """
    try:
        # Ejecutamos la inferencia con YOLO
        # conf=0.5 filtra detecciones con menos del 50% de certeza para mayor precisión
        resultados = yolo_model(ruta_imagen, conf=0.5, verbose=False)

        # Procesamos los resultados
        objetos_encontrados = []

        # Tomamos el primer resultado (solo hay una imagen)
        result = resultados[0]

        if not result.boxes:
            return "No se han detectado objetos reconocibles en la imagen"

        # Extraemos los nombres de las clases detectadas
        for box in result.boxes:
            cls_id = int(box.cls[0])
            nombre_clase = result.names[cls_id]
            objetos_encontrados.append(nombre_clase)

        # Contamos los objetos (ej: {'person': 2, 'dog': 1})
        conteo = Counter(objetos_encontrados)

        # Formateamos la respuesta para el agente
        descripcion = "En la imagen se han detectado: " + ", ".join([f"{cant} {obj}" for obj, cant in conteo.items()])
        return descripcion

    except Exception as e:
        return f"Error al procesar la imagen: {str(e)}"

# Lista de herramientas para el agente
tools = [detectar_objetos]

In [5]:
# Suprimimos los warnings de versiones para mantener la salida limpia
warnings.filterwarnings("ignore")

# Inicializamos el LLM con Groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Creamos el agente ReAct
# Este agente sabrá automáticamente cuándo llamar a 'detectar_objetos'
agent_executor = create_react_agent(llm, tools)

print("Agente creado exitosamente")

Agente creado exitosamente


In [6]:
def main():
    print("Sube tu imagen")
    # Función de Colab para subir archivos
    uploaded = files.upload()

    if not uploaded:
        print("No se subió ninguna imagen.")
        return

    # Obtenemos el nombre del archivo subido
    filename = next(iter(uploaded))
    print(f"\nImagen '{filename}' recibida.")

    # Mostramos la imagen para confirmar
    display(Image(filename, width=400))

    print("\nPregunta al Agente")
    pregunta_usuario = input("¿Qué quieres saber sobre la imagen? (ej: ¿Qué ves aquí?): ")

    if not pregunta_usuario:
        pregunta_usuario = "¿Qué objetos hay en esta imagen?"

    # Construimos el prompt. Es CRUCIAL decirle al agente dónde está la imagen (filename) para que sepa qué pasarle a la herramienta 'detectar_objetos'.
    mensaje_sistema = f"El usuario ha subido una imagen llamada '{filename}'. {pregunta_usuario} Usa la herramienta de detección si es necesario."

    print("\n[Agente Pensando...]\n")

    # Ejecutamos el agente
    response = agent_executor.invoke({"messages": [("user", mensaje_sistema)]})

    # Imprimimos la última respuesta del agente
    print("-" * 50)
    print("RESPUESTA DEL AGENTE:")
    print(response["messages"][-1].content)
    print("-" * 50)

# Ejecutar la aplicación
if __name__ == "__main__":
    main()

Sube tu imagen


KeyboardInterrupt: 